In [ ]:
import joblib
import re
import json
import os
from sentence_transformers import SentenceTransformer

try:
    # Laden des statischen Vektorraum-Modells (SBERT)
    mein_sbert = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    # Laden des trainierten Klassifikationsmodells aus der Persistierungsschicht
    mein_fertiges_modell = joblib.load('trainiertes_Klassifikationsmodell_SBERT.joblib')
except FileNotFoundError:
    print("FEHLER: Modell-Datei (.joblib) nicht gefunden. Bitte zuerst das Training abschließen!")
    exit()

CONFIDENCE_THRESHOLD = 0.40  


# --- 1. Daten laden ---

with open("Testmeldung.json", "r" , encoding="utf-8") as f_in:
    meldung = json.load(f_in)


if isinstance(meldung, list):
    payload = meldung[0]
else:
    payload = meldung

text = payload.get("description")
service_name = payload.get("service_name")  # Die zuvor ausgewählte Kategorie


# Preprocessing
def preprocessing_sbert(text):
    if not isinstance(text, str) or text.strip() == '':
        return ""
    # Nur Zeilenumbrüche durch Leerzeichen ersetzen
    text = re.sub(r'[\r\n]+', ' ', text)
    # Doppelte Leerzeichen entfernen
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


verarbeiteter_text = preprocessing_sbert(text)
vektor = mein_sbert.encode([verarbeiteter_text])

# --- 2. Vorhersage & Wahrscheinlichkeiten ---

vorhersage = mein_fertiges_modell.predict(vektor)
wahrscheinlichkeiten = mein_fertiges_modell.predict_proba(vektor)[0]

kategorien = mein_fertiges_modell.classes_

# Zusammenführen und sortieren (Höchste Wahrscheinlichkeit zuerst)
verteilung = dict(zip(kategorien, wahrscheinlichkeiten))
sortierte_verteilung = sorted(verteilung.items(), key=lambda x: x[1], reverse=True)

# Extraktion der Top-Kandidaten für die Margin-Berechnung
top1_kat, top1_prob = sortierte_verteilung[0]
top2_kat, top2_prob = sortierte_verteilung[1] if len(sortierte_verteilung) > 1 else ("Keine", 0.0)


# --- 3. Regelbasierte Routing-Logik (Human-in-the-Loop) ---

# OPTION A: Mathematische Margin (Differenz Top 1 zu Top 2)
confidence_margin = top1_prob - top2_prob

# OPTION B: Falls Kap. 4.4 nur den reinen Top-1 Wert meint, aktiviere diese Zeile:
# confidence_margin = top1_prob


print("="*50)
print(f"Meldungstext: {text}")
print(f"Von der Meldenden Person ausgewählte Kategorie: {service_name}")
print(f"Top-1 Vorhersage (Modell):           {top1_kat} ({top1_prob * 100:.2f}%)")
print(f"Top-2 Vorhersage (Modell):           {top2_kat} ({top2_prob * 100:.2f}%)")
print("="*50)
print(f"Berechnete Margin:                  {confidence_margin:.4f} (Schwellenwert: {CONFIDENCE_THRESHOLD})")
print("-" * 50)

if confidence_margin > CONFIDENCE_THRESHOLD:
    print(f"STATUS: AUTOMATISIERTE ZUWEISUNG")
    print(f"Ergebnis: Die Meldung wird direkt an die Fachabteilung '{top1_kat}' geroutet.")
    
    # --- NEU: Prüfung auf Fehlinterpretation / Abweichung bei hoher Konfidenz ---
    if service_name and top1_kat != service_name:
        print(f" Die zuvor gewählte Kategorie war mit einer Confidence-Margin von: {confidence_margin:.4f} falsch")
        print(f"  - Vorher ausgewählt:    '{service_name}'")
        print(f"  - Modell-Entscheidung:  '{top1_kat}'")


else:
    print(f"STATUS: RÜCKWEISUNG (Human-in-the-Loop)")
    print(f"Ergebnis: Schwellenwert unterschritten. Übergabe an Sachbearbeiter.")
    print("\nVerfügbare Top-3 Kategorievorschläge:")
    
    # Extrahiere exakt die Top-3 (oder weniger, falls das Modell nicht so viele hat)
    top_3_vorschlaege = sortierte_verteilung[:3]
    for i, (kat, prob) in enumerate(top_3_vorschlaege, 1):
        print(f"  {i}. {kat:<30} : {prob * 100:>6.2f}%")

print("="*50)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Meldungstext Die Leuchte ist ausgefallen Grünstreifen verbrannt
Von der Meldenden Person ausgewählte Kategorie (JSON): Leuchte ausgefallen
Top-1 Kategorie (Modell):           Leuchte beschädigt (41.33%)
Top-2 Kategorie (Modell):           Leuchte ausgefallen (24.62%)
Berechnete Margin:                  0.1671 (Schwellenwert: 0.4)
--------------------------------------------------
STATUS: RÜCKWEISUNG (Human-in-the-Loop)
Ergebnis: Schwellenwert unterschritten. Übergabe an Sachbearbeiter.

Verfügbare Top-3 Kategorievorschläge:
  1. Leuchte beschädigt             :  41.33%
  2. Leuchte ausgefallen            :  24.62%
  3. Park verschmutzt               :  19.38%
